**Step 1 | Install and initialize PySpark**

In [44]:
!pip -q install pyspark

In [45]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from pyspark import StorageLevel
import time
spark = SparkSession.builder \
.appName("PySparkPractice") \
.getOrCreate()

**Step 2 | Create product and store DataFrames**

In [46]:
products_data = [
    ("P01", "Laptop", "Electronics", 60000),
    ("P02", "Monitor", "Electronics", 14000),
    ("P03", "Headphones", "Accessories", 3500),
    ("P04", "Printer", "Office", 17000),
    ("P05", "Router", "Networking", 4500),
    ("P06", "Tablet", "Electronics", 24000)
]
products_df = spark.createDataFrame(
    products_data,
    ["product_id", "product_name", "category", "list_price"]
)
stores_data = [
    ("S01", "Hyderabad", "South"),
    ("S02", "Bengaluru", "South"),
    ("S03", "Mumbai", "West"),
    ("S04", "Delhi", "North")
]
stores_df = spark.createDataFrame(
    stores_data,
    ["store_id", "city", "region"]
)
products_df.show()
stores_df.show()

+----------+------------+-----------+----------+
|product_id|product_name|   category|list_price|
+----------+------------+-----------+----------+
|       P01|      Laptop|Electronics|     60000|
|       P02|     Monitor|Electronics|     14000|
|       P03|  Headphones|Accessories|      3500|
|       P04|     Printer|     Office|     17000|
|       P05|      Router| Networking|      4500|
|       P06|      Tablet|Electronics|     24000|
+----------+------------+-----------+----------+

+--------+---------+------+
|store_id|     city|region|
+--------+---------+------+
|     S01|Hyderabad| South|
|     S02|Bengaluru| South|
|     S03|   Mumbai|  West|
|     S04|    Delhi| North|
+--------+---------+------+



**Step 3 | Generate 100,000 sales records**

In [47]:
sales_df = (
    spark.range(1, 100001)
    .withColumnRenamed("id", "order_id")
    .withColumn(
        "order_date",
        F.date_add(
            F.to_date(F.lit("2026-01-01")),
            (F.col("order_id") % 180).cast("int")
        )
    )
    .withColumn(
        "product_id",
        F.concat(
            F.lit("P"),
            F.lpad(
                ((F.col("order_id") % 6) + 1).cast("string"),
                2, "0"
            )
        )
    )
    .withColumn(
        "store_id",
        F.concat(
            F.lit("S"),
            F.lpad(
                ((F.floor(F.col("order_id") / 3) % 4) + 1)
                .cast("string"),
                2, "0"
            )
        )
    )
    .withColumn(
        "quantity",
        ((F.col("order_id") % 8) + 1).cast("int")
    )
    .withColumn(
        "discount_pct",
        F.when(
            F.col("order_id") % 19 == 0,
            F.lit(None).cast("int")
        ).otherwise(
            ((F.floor(F.col("order_id") / 5) % 4) * 5)
            .cast("int")
        )
    )
    .withColumn(
        "order_status",
        F.when(F.col("order_id") % 11 == 0, "Cancelled")
         .when(F.col("order_id") % 7 == 0, "Returned")
         .otherwise("Completed")
    )

    .withColumn(
        "payment_method",
        F.when(F.col("order_id") % 3 == 0, "UPI")
         .when(F.col("order_id") % 3 == 1, "Card")
         .otherwise("Cash")
    )

    .withColumn(
        "salesperson_id",
        F.when(
            F.col("order_id") % 17 == 0,
            F.lit(None).cast("string")
        ).otherwise(
            F.concat(
                F.lit("EMP"),
                F.lpad(
                    ((F.col("order_id") % 12) + 1).cast("string"),
                    2, "0"
                )
            )
        )
    )

    .join(
        products_df.select(
            "product_id",
            F.col("list_price").alias("unit_price")
        ),
        "product_id",
        "left"
    )

    .select(
        "order_id", "order_date", "store_id",
        "product_id", "quantity", "unit_price",
        "discount_pct", "order_status",
        "payment_method", "salesperson_id"
    )
)

print("Total Records:", sales_df.count())
sales_df.printSchema()
sales_df.show(10, truncate=False)


Total Records: 100000
root
 |-- order_id: long (nullable = false)
 |-- order_date: date (nullable = true)
 |-- store_id: string (nullable = true)
 |-- product_id: string (nullable = true)
 |-- quantity: integer (nullable = true)
 |-- unit_price: long (nullable = true)
 |-- discount_pct: integer (nullable = true)
 |-- order_status: string (nullable = false)
 |-- payment_method: string (nullable = false)
 |-- salesperson_id: string (nullable = true)

+--------+----------+--------+----------+--------+----------+------------+------------+--------------+--------------+
|order_id|order_date|store_id|product_id|quantity|unit_price|discount_pct|order_status|payment_method|salesperson_id|
+--------+----------+--------+----------+--------+----------+------------+------------+--------------+--------------+
|3       |2026-01-04|S02     |P04       |4       |17000     |0           |Completed   |UPI           |EMP04         |
|9       |2026-01-10|S04     |P04       |2       |17000     |5           |C

# **Practice Exercise**

**Part A | DataFrame transformations**

In [48]:
# 1. Select order_id, product_id, quantity, and unit_price from the sales DataFrame.
sales_df.select("order_id", "product_id", "quantity", "unit_price").show()

+--------+----------+--------+----------+
|order_id|product_id|quantity|unit_price|
+--------+----------+--------+----------+
|       3|       P04|       4|     17000|
|       9|       P04|       2|     17000|
|      15|       P04|       8|     17000|
|      21|       P04|       6|     17000|
|       2|       P03|       3|      3500|
|       8|       P03|       1|      3500|
|      14|       P03|       7|      3500|
|      20|       P03|       5|      3500|
|       4|       P05|       5|      4500|
|      10|       P05|       3|      4500|
|      16|       P05|       1|      4500|
|       5|       P06|       6|     24000|
|      11|       P06|       4|     24000|
|      17|       P06|       2|     24000|
|       1|       P02|       2|     14000|
|       7|       P02|       8|     14000|
|      13|       P02|       6|     14000|
|      19|       P02|       4|     14000|
|       6|       P01|       7|     60000|
|      12|       P01|       5|     60000|
+--------+----------+--------+----

In [49]:
# 2. Filter all completed transactions where quantity is greater than 5.
sales_df.filter((sales_df.order_status == "Completed") & (sales_df.quantity > 5)).show()

+--------+----------+--------+----------+--------+----------+------------+------------+--------------+--------------+
|order_id|order_date|store_id|product_id|quantity|unit_price|discount_pct|order_status|payment_method|salesperson_id|
+--------+----------+--------+----------+--------+----------+------------+------------+--------------+--------------+
|      15|2026-01-16|     S02|       P04|       8|     17000|          15|   Completed|           UPI|         EMP04|
|      39|2026-02-09|     S02|       P04|       8|     17000|          15|   Completed|           UPI|         EMP04|
|      45|2026-02-15|     S04|       P04|       6|     17000|           5|   Completed|           UPI|         EMP10|
|      69|2026-03-11|     S04|       P04|       6|     17000|           5|   Completed|           UPI|         EMP10|
|      38|2026-02-08|     S01|       P03|       7|      3500|        NULL|   Completed|          Cash|         EMP03|
|      62|2026-03-04|     S01|       P03|       7|      

In [50]:
# 3. Create a column named gross_revenue by multiplying quantity and unit_price.
sales_df.withColumn("gross_revenue", F.col("quantity") * F.col("unit_price")).show()

+--------+----------+--------+----------+--------+----------+------------+------------+--------------+--------------+-------------+
|order_id|order_date|store_id|product_id|quantity|unit_price|discount_pct|order_status|payment_method|salesperson_id|gross_revenue|
+--------+----------+--------+----------+--------+----------+------------+------------+--------------+--------------+-------------+
|       3|2026-01-04|     S02|       P04|       4|     17000|           0|   Completed|           UPI|         EMP04|        68000|
|       9|2026-01-10|     S04|       P04|       2|     17000|           5|   Completed|           UPI|         EMP10|        34000|
|      15|2026-01-16|     S02|       P04|       8|     17000|          15|   Completed|           UPI|         EMP04|       136000|
|      21|2026-01-22|     S04|       P04|       6|     17000|           0|    Returned|           UPI|         EMP10|       102000|
|       2|2026-01-03|     S01|       P03|       3|      3500|           0|  

In [51]:
# 4. Rename order_status to transaction_status, then drop the payment_method column.
sales_df.withColumnRenamed("order_status", "transaction_Status").show()
drop_df = sales_df.drop("payment_method")
drop_df.show()

+--------+----------+--------+----------+--------+----------+------------+------------------+--------------+--------------+
|order_id|order_date|store_id|product_id|quantity|unit_price|discount_pct|transaction_Status|payment_method|salesperson_id|
+--------+----------+--------+----------+--------+----------+------------+------------------+--------------+--------------+
|       3|2026-01-04|     S02|       P04|       4|     17000|           0|         Completed|           UPI|         EMP04|
|       9|2026-01-10|     S04|       P04|       2|     17000|           5|         Completed|           UPI|         EMP10|
|      15|2026-01-16|     S02|       P04|       8|     17000|          15|         Completed|           UPI|         EMP04|
|      21|2026-01-22|     S04|       P04|       6|     17000|           0|          Returned|           UPI|         EMP10|
|       2|2026-01-03|     S01|       P03|       3|      3500|           0|         Completed|          Cash|         EMP03|
|       

**Part B | Aggregations and joins**

In [52]:
# 5. Calculate the total quantity sold for each product.
sales_product_df = sales_df.join(products_df, on = "product_id", how = "left")
sales_product_df.groupBy("product_name").agg(F.sum("quantity").alias("total_quantity_sold")).show()

+------------+-------------------+
|product_name|total_quantity_sold|
+------------+-------------------+
|      Router|              66665|
|      Laptop|              66668|
|      Tablet|              83330|
|     Printer|              83334|
|     Monitor|              83336|
|  Headphones|              66667|
+------------+-------------------+



In [53]:
# 6. Join sales with product master data and calculate total gross revenue by product category.
sales_product_df.groupBy("category").agg(F.sum("quantity").alias("total_quantity_sold")).show()

+-----------+-------------------+
|   category|total_quantity_sold|
+-----------+-------------------+
|     Office|              83334|
| Networking|              66665|
|Electronics|             233334|
|Accessories|              66667|
+-----------+-------------------+



In [54]:
# 7. Join sales with store master data and calculate total transactions for each city.
sales_product_df = sales_df.join(stores_df, on = "store_id", how = "left")
sales_product_df.groupBy("city").agg(F.count("order_id").alias("total_transactions")).show()

+---------+------------------+
|     city|total_transactions|
+---------+------------------+
|   Mumbai|             24999|
|    Delhi|             24999|
|Bengaluru|             25001|
|Hyderabad|             25001|
+---------+------------------+



In [55]:
# 8. Join all three DataFrames and find the top five products by gross revenue for completed orders.
all_info_df = sales_df.join(products_df, on="product_id", how="inner") \
              .join(stores_df, on="store_id", how="inner") \
              .withColumn("gross_revenue", F.col("quantity") * F.col("unit_price"))
all_info_df.filter(all_info_df.order_status == "Completed").sort(F.col("gross_revenue").desc()).show(5)

+--------+----------+--------+----------+--------+----------+------------+------------+--------------+--------------+------------+-----------+----------+------+------+-------------+
|store_id|product_id|order_id|order_date|quantity|unit_price|discount_pct|order_status|payment_method|salesperson_id|product_name|   category|list_price|  city|region|gross_revenue|
+--------+----------+--------+----------+--------+----------+------------+------------+--------------+--------------+------------+-----------+----------+------+------+-------------+
|     S03|       P01|       6|2026-01-07|       7|     60000|           5|   Completed|           UPI|         EMP07|      Laptop|Electronics|     60000|Mumbai|  West|       420000|
|     S03|       P01|     150|2026-05-31|       7|     60000|          10|   Completed|           UPI|         EMP07|      Laptop|Electronics|     60000|Mumbai|  West|       420000|
|     S03|       P01|      30|2026-01-31|       7|     60000|          10|   Completed|   

**Part C | Null handling and conditions**

In [56]:
# 9. Count null values in discount_pct and salesperson_id.
sales_df.filter((F.col("discount_pct").isNull()) & F.col("salesperson_id").isNull()).count()

309

In [57]:
# 10. Replace null discounts with zero using na.fill().
sales_df.fillna({"discount_pct": 0}).show()

+--------+----------+--------+----------+--------+----------+------------+------------+--------------+--------------+
|order_id|order_date|store_id|product_id|quantity|unit_price|discount_pct|order_status|payment_method|salesperson_id|
+--------+----------+--------+----------+--------+----------+------------+------------+--------------+--------------+
|       3|2026-01-04|     S02|       P04|       4|     17000|           0|   Completed|           UPI|         EMP04|
|       9|2026-01-10|     S04|       P04|       2|     17000|           5|   Completed|           UPI|         EMP10|
|      15|2026-01-16|     S02|       P04|       8|     17000|          15|   Completed|           UPI|         EMP04|
|      21|2026-01-22|     S04|       P04|       6|     17000|           0|    Returned|           UPI|         EMP10|
|       2|2026-01-03|     S01|       P03|       3|      3500|           0|   Completed|          Cash|         EMP03|
|       8|2026-01-09|     S03|       P03|       1|      

In [58]:
# 11. Remove records where salesperson_id is null using na.drop().
sales_df.dropna(subset=["salesperson_id"]).show()

+--------+----------+--------+----------+--------+----------+------------+------------+--------------+--------------+
|order_id|order_date|store_id|product_id|quantity|unit_price|discount_pct|order_status|payment_method|salesperson_id|
+--------+----------+--------+----------+--------+----------+------------+------------+--------------+--------------+
|       3|2026-01-04|     S02|       P04|       4|     17000|           0|   Completed|           UPI|         EMP04|
|       9|2026-01-10|     S04|       P04|       2|     17000|           5|   Completed|           UPI|         EMP10|
|      15|2026-01-16|     S02|       P04|       8|     17000|          15|   Completed|           UPI|         EMP04|
|      21|2026-01-22|     S04|       P04|       6|     17000|           0|    Returned|           UPI|         EMP10|
|       2|2026-01-03|     S01|       P03|       3|      3500|           0|   Completed|          Cash|         EMP03|
|       8|2026-01-09|     S03|       P03|       1|      

In [59]:
# 12. Create order_size using when() and otherwise(): Small for quantity 1-3, Medium for 4-6, and Large for 7 or more.
sales_df.withColumn(
    "order_size",
    F.when((F.col("quantity") >= 1) & (F.col("quantity") <= 3), "Small")
     .when((F.col("quantity") >= 4) & (F.col("quantity") <= 6), "Medium")
     .when(F.col("quantity") >= 7, "Large")
     .otherwise("Unknown")
).show()

+--------+----------+--------+----------+--------+----------+------------+------------+--------------+--------------+----------+
|order_id|order_date|store_id|product_id|quantity|unit_price|discount_pct|order_status|payment_method|salesperson_id|order_size|
+--------+----------+--------+----------+--------+----------+------------+------------+--------------+--------------+----------+
|       3|2026-01-04|     S02|       P04|       4|     17000|           0|   Completed|           UPI|         EMP04|    Medium|
|       9|2026-01-10|     S04|       P04|       2|     17000|           5|   Completed|           UPI|         EMP10|     Small|
|      15|2026-01-16|     S02|       P04|       8|     17000|          15|   Completed|           UPI|         EMP04|     Large|
|      21|2026-01-22|     S04|       P04|       6|     17000|           0|    Returned|           UPI|         EMP10|    Medium|
|       2|2026-01-03|     S01|       P03|       3|      3500|           0|   Completed|          

**Part D | Window and date functions**

In [60]:
from pyspark.sql.window import Window
from pyspark.sql.functions import (row_number,rank,dense_rank)

In [73]:
# 13. Rank products by total gross revenue within each category using dense_rank().
product_rev_df = all_info_df.groupBy("category", "product_name") \
    .agg(F.sum("gross_revenue").alias("total_gross_revenue"))
window_rank = Window.partitionBy("category").orderBy(F.desc("total_gross_revenue"))

df_ranked = product_rev_df.withColumn("dense rank", F.dense_rank().over(window_rank))

df_ranked.show()

+-----------+------------+-------------------+----------+
|   category|product_name|total_gross_revenue|dense rank|
+-----------+------------+-------------------+----------+
|Accessories|  Headphones|          233334500|         1|
|Electronics|      Laptop|         4000080000|         1|
|Electronics|      Tablet|         1999920000|         2|
|Electronics|     Monitor|         1166704000|         3|
| Networking|      Router|          299992500|         1|
|     Office|     Printer|         1416678000|         1|
+-----------+------------+-------------------+----------+



In [62]:
# 14. Calculate running gross revenue by store, ordered by date and order ID.
window_running = Window.partitionBy("store_id").orderBy("order_date", "order_id")
df_running_total = all_info_df.withColumn("running_gross_revenue", F.sum("gross_revenue").over(window_running))

df_running_total.select("store_id", "order_date", "order_id", "gross_revenue", "running_gross_revenue").show()

+--------+----------+--------+-------------+---------------------+
|store_id|order_date|order_id|gross_revenue|running_gross_revenue|
+--------+----------+--------+-------------+---------------------+
|     S01|2026-01-01|     180|       300000|               300000|
|     S01|2026-01-01|     360|        60000|               360000|
|     S01|2026-01-01|     540|       300000|               660000|
|     S01|2026-01-01|     720|        60000|               720000|
|     S01|2026-01-01|     900|       300000|              1020000|
|     S01|2026-01-01|    1080|        60000|              1080000|
|     S01|2026-01-01|    1260|       300000|              1380000|
|     S01|2026-01-01|    1440|        60000|              1440000|
|     S01|2026-01-01|    1620|       300000|              1740000|
|     S01|2026-01-01|    1800|        60000|              1800000|
|     S01|2026-01-01|    1980|       300000|              2100000|
|     S01|2026-01-01|    2160|        60000|              2160

In [63]:
# 15. Extract the month from order_date and calculate monthly sales revenue for completed orders.
df_monthly = all_info_df.filter(F.col("order_status") == "Completed") \
    .withColumn("order_month", F.month(F.to_date("order_date"))) \
    .groupBy("order_month") \
    .agg(F.sum("gross_revenue").alias("monthly_sales_revenue"))

df_monthly.show()

+-----------+---------------------+
|order_month|monthly_sales_revenue|
+-----------+---------------------+
|          1|           1278728500|
|          6|           1089027500|
|          3|           1212087500|
|          5|           1277560500|
|          4|           1218921000|
|          2|           1027585500|
+-----------+---------------------+



**Part E | Lazy execution**

In [64]:
# 16. Create three consecutive transformations: filter(), withColumn(), and select(). Add a Python print() after each transformation. Trigger execution only at the end using show().
df_trans1 = sales_df.filter(F.col("order_status") == "Completed")
print("Filtered records with order status completed.")

df_trans2 = df_trans1.withColumn("discount_amount", F.col("unit_price") * F.col("discount_pct") / 100)
print("Added a column discount amount.")

df_trans3 = df_trans2.select("order_id", "salesperson_id", "order_status", "discount_amount")
print("Selected order id, salesperson id, order status, discount amount.")

df_trans3.show()

Filtered records with order status completed.
Added a column discount amount.
Selected order id, salesperson id, order status, discount amount.
+--------+--------------+------------+---------------+
|order_id|salesperson_id|order_status|discount_amount|
+--------+--------------+------------+---------------+
|       3|         EMP04|   Completed|            0.0|
|       9|         EMP10|   Completed|          850.0|
|      15|         EMP04|   Completed|         2550.0|
|       2|         EMP03|   Completed|            0.0|
|       8|         EMP09|   Completed|          175.0|
|      20|         EMP09|   Completed|            0.0|
|      26|         EMP03|   Completed|          175.0|
|       4|         EMP05|   Completed|            0.0|
|      10|         EMP11|   Completed|          450.0|
|      16|         EMP05|   Completed|          675.0|
|       5|         EMP06|   Completed|         1200.0|
|      17|          NULL|   Completed|         3600.0|
|      23|         EMP12|   Com

In [65]:
# 17. Create an aggregation by product_id. Use explain() to inspect the execution plan before calling collect().
df_agg = sales_df.groupBy("product_id").agg(F.count("order_id").alias("total_orders"))

df_agg.explain()
collected_data = df_agg.collect()
print(f"Collected Result: {collected_data}")

== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- HashAggregate(keys=[product_id#2144], functions=[count(1)])
   +- HashAggregate(keys=[product_id#2144], functions=[partial_count(1)])
      +- Project [product_id#2144]
         +- SortMergeJoin [product_id#2144], [product_id#2111], LeftOuter
            :- Sort [product_id#2144 ASC NULLS FIRST], false, 0
            :  +- Exchange hashpartitioning(product_id#2144, 200), ENSURE_REQUIREMENTS, [plan_id=15797]
            :     +- Project [concat(P, lpad(cast(((id#2141L % 6) + 1) as string), 2, 0)) AS product_id#2144]
            :        +- Range (1, 100001, step=1, splits=2)
            +- Sort [product_id#2111 ASC NULLS FIRST], false, 0
               +- Exchange hashpartitioning(product_id#2111, 200), ENSURE_REQUIREMENTS, [plan_id=15798]
                  +- Project [product_id#2111]
                     +- Filter isnotnull(product_id#2111)
                        +- Scan ExistingRDD[product_id#2111,product_name#2112,category#

**Part F | Caching and persistence**

In [66]:
# 18. Filter completed orders, call cache(), and execute count() twice. Measure the execution time of each action.
df_completed = sales_df.filter(F.col("order_status") == "Completed").cache()

# First execution
start = time.perf_counter()

df_completed.count()

time1 = time.perf_counter() - start

# Second execution
start = time.perf_counter()

df_completed.count()

time2 = time.perf_counter() - start

print("First Execution:", round(time1, 3), "seconds")
print("Second Execution:", round(time2, 3), "seconds")

First Execution: 5.614 seconds
Second Execution: 1.523 seconds


In [67]:
# 19. Persist transactions with quantity greater than 5 using StorageLevel.MEMORY_AND_DISK. Execute two actions, then release the persisted data using unpersist().
high_quantity_df = sales_df.filter(
F.col("quantity") > 5
)

# Persist using MEMORY_AND_DISK
high_quantity_df.persist(
StorageLevel.MEMORY_AND_DISK
)

# Trigger executions
high_quantity_df.count()
high_quantity_df.show(10)

# Check storage level
print(high_quantity_df.storageLevel)

# Check cache status
print("Before:", high_quantity_df.is_cached)

# Remove cache
high_quantity_df.unpersist(blocking=True)

# Check again
print("After:", high_quantity_df.is_cached)

+--------+----------+--------+----------+--------+----------+------------+------------+--------------+--------------+
|order_id|order_date|store_id|product_id|quantity|unit_price|discount_pct|order_status|payment_method|salesperson_id|
+--------+----------+--------+----------+--------+----------+------------+------------+--------------+--------------+
|      15|2026-01-16|     S02|       P04|       8|     17000|          15|   Completed|           UPI|         EMP04|
|      21|2026-01-22|     S04|       P04|       6|     17000|           0|    Returned|           UPI|         EMP10|
|      39|2026-02-09|     S02|       P04|       8|     17000|          15|   Completed|           UPI|         EMP04|
|      45|2026-02-15|     S04|       P04|       6|     17000|           5|   Completed|           UPI|         EMP10|
|      63|2026-03-05|     S02|       P04|       8|     17000|           0|    Returned|           UPI|         EMP04|
|      69|2026-03-11|     S04|       P04|       6|     1

In [68]:
# 20. Cache an aggregated DataFrame, inspect its execution plan, and identify InMemoryTableScan or InMemoryRelation.
df_cached_agg = sales_df.groupBy("product_id").agg(F.avg("unit_price")).cache()
df_cached_agg.count()

df_cached_agg.explain()

== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- InMemoryTableScan [product_id#2144, avg(unit_price)#3853]
      +- InMemoryRelation [product_id#2144, avg(unit_price)#3853], StorageLevel(disk, memory, deserialized, 1 replicas)
            +- AdaptiveSparkPlan isFinalPlan=true
               +- == Final Plan ==
                  ResultQueryStage 3
                  +- *(3) HashAggregate(keys=[product_id#2144], functions=[avg(unit_price#2151L)])
                     +- *(3) HashAggregate(keys=[product_id#2144], functions=[partial_avg(unit_price#2151L)])
                        +- *(3) Project [product_id#2144, unit_price#2151L]
                           +- *(3) BroadcastHashJoin [product_id#2144], [product_id#2111], LeftOuter, BuildRight, false
                              :- ShuffleQueryStage 0
                              :  +- Exchange hashpartitioning(product_id#2144, 200), ENSURE_REQUIREMENTS, [plan_id=16381]
                              :     +- *(1) Project [concat(P

**Part G | Repartition and coalesce**

In [69]:
# 21. Check the existing partition count, then repartition the sales DataFrame into 8 partitions.
print("Original Partitions:",
sales_df.rdd.getNumPartitions())

partitioned_df = sales_df.repartition(8)

print("New Partitions:",
partitioned_df.rdd.getNumPartitions())

Original Partitions: 2
New Partitions: 8


In [70]:
# 22. Display the number of records in each partition using spark_partition_id().
partitioned_df.withColumn(
"partition_id",
F.spark_partition_id()
).groupBy(
"partition_id"
).count().orderBy(
"partition_id"
).show()

+------------+-----+
|partition_id|count|
+------------+-----+
|           0|12500|
|           1|12500|
|           2|12500|
|           3|12500|
|           4|12500|
|           5|12500|
|           6|12500|
|           7|12500|
+------------+-----+



In [71]:
# 23. Repartition the sales DataFrame into 6 partitions based on store_id. Inspect which stores appear in each partition.
store_partitioned_df = sales_df.repartition(6,"store_id")
print("Partitions:", store_partitioned_df.rdd.getNumPartitions())

store_partitioned_df.withColumn(
"partition_id",
F.spark_partition_id()
).groupBy(
"partition_id", "store_id"
).count().orderBy(
"partition_id", "store_id"
).show()

Partitions: 6
+------------+--------+-----+
|partition_id|store_id|count|
+------------+--------+-----+
|           1|     S03|24999|
|           3|     S02|25001|
|           4|     S01|25001|
|           5|     S04|24999|
+------------+--------+-----+



In [74]:
# 24. Create a DataFrame with 12 partitions, then reduce it to 4 using coalesce(). Compare its execution plan with repartition(4).
df_12 = store_partitioned_df.repartition(12)
print("Partitions:", df_12.rdd.getNumPartitions())

df_4 = df_12.coalesce(4)
print("Partitions:", df_4.rdd.getNumPartitions())

df_12.explain()
df_4.explain()

Partitions: 12
Partitions: 4
== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=true
+- == Final Plan ==
   ResultQueryStage 4
   +- ShuffleQueryStage 3
      +- Exchange RoundRobinPartitioning(12), REPARTITION_BY_NUM, [plan_id=19307]
         +- *(3) Project [order_id#2142L, order_date#2143, store_id#2145, product_id#2144, quantity#2146, unit_price#2151L, discount_pct#2147, order_status#2148, payment_method#2149, salesperson_id#2150]
            +- *(3) BroadcastHashJoin [product_id#2144], [product_id#2111], LeftOuter, BuildRight, false
               :- AQEShuffleRead local
               :  +- ShuffleQueryStage 0
               :     +- Exchange hashpartitioning(product_id#2144, 200), ENSURE_REQUIREMENTS, [plan_id=19219]
               :        +- *(1) Project [id#2141L AS order_id#2142L, date_add(2026-01-01, cast((id#2141L % 180) as int)) AS order_date#2143, concat(P, lpad(cast(((id#2141L % 6) + 1) as string), 2, 0)) AS product_id#2144, concat(S, lpad(cast(((FLOOR((cast(id#2141L as 

In [75]:
# 25. Save the sales data into 4 CSV part files using repartition(4) and write.csv().
sales_df.repartition(4).write.mode("overwrite").option("header", "true").csv("output/sales_data_csv")

**Final Challenge**

In [78]:
# 1. Filter completed orders
completed_sales = sales_df.filter(F.col("order_status") == "Completed")

# 2. Replace null discounts with zero
completed_sales = completed_sales.fillna({"discount_pct": 0})

# 3. Join product and store details
sales_details = completed_sales.join(products_df, on="product_id", how="left") \
                               .join(stores_df, on="store_id", how="left")

# 4. Calculate gross revenue and discounted revenue
sales_details = sales_details.withColumn(
    "gross_revenue",
    F.col("quantity") * F.col("unit_price")
).withColumn(
    "discounted_revenue",
    F.col("gross_revenue") * (1 - F.col("discount_pct"))
)

# 5. Group by region and product category
# 6. Calculate total revenue and transaction count
revenue_summary = sales_details.groupBy("region", "category").agg(
    F.sum("discounted_revenue").alias("total_revenue"),
    F.count("*").alias("transaction_count")
)

# 7. Rank categories by revenue within each region
window_spec = Window.partitionBy("region").orderBy(
    F.col("total_revenue").desc()
)

final_df = revenue_summary.withColumn(
    "revenue_rank",
    F.rank().over(window_spec)
)

# 8. Cache the final aggregated DataFrame
final_df.cache()

# 9. Display the results
final_df.orderBy("region", "revenue_rank").show(truncate=False)

# 10. Write output into 2 Parquet part files
final_df.coalesce(2).write.mode("overwrite").parquet("output/sales_summary")

+------+-----------+-------------+-----------------+------------+
|region|category   |total_revenue|transaction_count|revenue_rank|
+------+-----------+-------------+-----------------+------------+
|North |Networking |-960925500   |6494             |1           |
|North |Office     |-2486012000  |6492             |2           |
|North |Electronics|-7031712000  |6492             |3           |
|South |Networking |-410818500   |6494             |1           |
|South |Accessories|-747666500   |6494             |2           |
|South |Office     |-4989296000  |6495             |3           |
|South |Electronics|-11039424000 |19484            |4           |
|West  |Accessories|-318930500   |6493             |1           |
|West  |Electronics|-16917412000 |12985            |2           |
+------+-----------+-------------+-----------------+------------+

